In [46]:
import numpy as np
import pandas as pd


In [47]:
df = pd.read_csv('Online Retail.csv')
print(df.head())

  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

      InvoiceDate  UnitPrice CustomerID         Country  
0  12/1/2010 8:26       2.55      17850  United Kingdom  
1  12/1/2010 8:26       3.39      17850  United Kingdom  
2  12/1/2010 8:26       2.75      17850  United Kingdom  
3  12/1/2010 8:26       3.39      17850  United Kingdom  
4  12/1/2010 8:26       3.39      17850  United Kingdom  


C:\Users\Darshini Kannan\AppData\Local\Temp\ipykernel_8128\2375839996.py:1: DtypeWarning: Columns (0: CustomerID) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('Online Retail.csv')


In [48]:
print({"rows": df.shape[0], "columns": df.shape[1]})

{'rows': 541909, 'columns': 8}


In [49]:
print(df.shape[0]), print(df.shape[1])

541909
8


(None, None)

In [50]:
print({
        "unique_CustomerID": int(df["CustomerID"].nunique()),
        "unique_rows_excluding_id": int(
            df.drop(columns=["CustomerID"]).astype(str).drop_duplicates().shape[0]
        ),
    }
)

{'unique_CustomerID': 5398, 'unique_rows_excluding_id': 536641}


In [65]:
counts = df.isna().sum()
print("Missing values:" ,{column: int(count) for column, count in counts.items() if count > 0} )


Missing values: {'Description': 1454, 'CustomerID': 135080}


In [62]:
print((df.isnull().mean() * 100).round(2))

InvoiceNo       0.00
StockCode       0.00
Description     0.27
Quantity        0.00
InvoiceDate     0.00
UnitPrice       0.00
CustomerID     24.93
Country         0.00
dtype: float64


In [52]:
columns = [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Country",
        "UnitPrice",
        "CustomerID",
    ]
print({column: int(df[column].nunique(dropna=True)) for column in columns})


{'InvoiceNo': 25900, 'StockCode': 4070, 'Description': 4223, 'Country': 38, 'UnitPrice': 1630, 'CustomerID': 5398}


In [66]:
print({
    "duplicate_rows": int(df.duplicated().sum())
})

{'duplicate_rows': 5268}


Number of customers appearing in more than one country:

In [79]:
per_Country = df.groupby("Country", observed=True)["CustomerID"].nunique()
print({"customers_in_multiple_countries": int((per_Country > 1).sum())})


{'customers_in_multiple_countries': 29}


In [68]:
cancelled = df["InvoiceNo"].astype(str).str.startswith("C")

print({
    "cancelled_rows": int(cancelled.sum()),
    "cancelled_invoices": int(df.loc[cancelled, "InvoiceNo"].nunique())
})

{'cancelled_rows': 9288, 'cancelled_invoices': 3836}


In [71]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

monthly_orders = (
    df.groupby(df["InvoiceDate"].dt.to_period("M"))["InvoiceNo"]
    .nunique()
)

print(monthly_orders)

InvoiceDate
2010-12    2025
2011-01    1476
2011-02    1393
2011-03    1983
2011-04    1744
2011-05    2162
2011-06    2012
2011-07    1927
2011-08    1737
2011-09    2327
2011-10    2637
2011-11    3462
2011-12    1015
Freq: M, Name: InvoiceNo, dtype: int64


In [ ]:
order_value = (
    df.groupby("InvoiceNo")
    .apply(lambda x: (x["Quantity"] * x["UnitPrice"]).sum())
    .reset_index(name="OrderValue")
)

average_order_value = order_value["OrderValue"].mean()

print({
    "average_order_value": round(average_order_value, 2)
})

{'average_order_value': np.float64(376.36)}


In [76]:
orders_per_customer = (
    df.groupby("CustomerID")["InvoiceNo"]
    .nunique()
    .sort_values(ascending=False)
)

print(orders_per_customer.head(20))

CustomerID
14911.0    228
12748.0    183
17841.0    153
14606.0    108
13089.0    103
15311.0     97
12971.0     78
14527.0     74
14646.0     72
13408.0     72
16422.0     67
16029.0     65
14156.0     63
18102.0     58
13694.0     55
13798.0     55
15189.0     52
17949.0     50
17450.0     50
17811.0     49
Name: InvoiceNo, dtype: int64


In [77]:
customer_orders = (
    df.groupby("CustomerID")["InvoiceNo"]
    .nunique()
)

repeat_customers = (customer_orders > 1).sum()

print({
    "repeat_customers": int(repeat_customers),
    "total_customers": int(customer_orders.shape[0])
})

{'repeat_customers': 3387, 'total_customers': 5398}


In [80]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

print(df["InvoiceDate"].min())
print(df["InvoiceDate"].max())

2010-12-01 08:26:00
2011-12-09 12:50:00


In [81]:
print(df["UnitPrice"].describe())

count    541909.000000
mean          4.611114
std          96.759853
min      -11062.060000
25%           1.250000
50%           2.080000
75%           4.130000
max       38970.000000
Name: UnitPrice, dtype: float64


In [82]:
print({
    "zero_unit_price": int((df["UnitPrice"] == 0).sum()),
    "negative_unit_price": int((df["UnitPrice"] < 0).sum()),
    "missing_unit_price": int(df["UnitPrice"].isnull().sum())
})

{'zero_unit_price': 2515, 'negative_unit_price': 2, 'missing_unit_price': 0}


In [99]:
print("rows:", df.shape[0])
print("columns:", df.shape[1])
print("duplicate_rows:", int(df.duplicated().sum()))
print("missing_description:", int(df["Description"].isnull().sum()))
print("missing_customer_id:", int(df["CustomerID"].isnull().sum()))
print("average_order_value:", round(average_order_value, 2))
print("invoice_date_min:", df["InvoiceDate"].min())
print("invoice_date_max:", df["InvoiceDate"].max())
print("cancelled_invoice:", int(df.loc[cancelled, "InvoiceNo"].nunique()))
print("zero_unit_price:", int((df["UnitPrice"] == 0).sum()))
print("negative_unit_price:", int((df["UnitPrice"] < 0).sum()))
print("cancelled_rows:", int(cancelled.sum()))
print("unique_customers:", int(df["CustomerID"].nunique()))

rows: 541909
columns: 8
duplicate_rows: 5268
missing_description: 1454
missing_customer_id: 135080
average_order_value: 376.36
invoice_date_min: 2010-12-01 08:26:00
invoice_date_max: 2011-12-09 12:50:00
cancelled_invoice: 3836
zero_unit_price: 2515
negative_unit_price: 2
cancelled_rows: 9288
unique_customers: 5398
